In [1]:
from pyspark import SparkConf
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("carpetSample").master("local[4]").getOrCreate()
sc = spark.sparkContext

In [2]:
df = spark.read.options(header=True, inferSchema=True, delimiter="\t").csv("hdfs://hdfs-namenode:9000/task1/input/customs_data.csv")

In [3]:
df.count()

26392290

In [4]:
dfSample = df.sample(fraction=0.01, seed=67069)

In [5]:
dfSample.count()

264105

In [6]:
from pyspark.sql.functions import col

In [7]:
dfSample.orderBy(col("month").desc()).show(n=67, truncate=False)

+----------+-------+---------+--------+-------+-------+-------+--------+------+-----------+
|code      |country|direction|district|measure|month  |netto  |quantity|region|value      |
+----------+-------+---------+--------+-------+-------+-------+--------+------+-----------+
|8701201013|NL     |ИМ       |1       |ШТ     |12/2021|55164  |7       |66000 |514669,15  |
|9403109809|PL     |ИМ       |2       |1      |12/2021|0      |0       |40000 |15918,37   |
|9401800009|KR     |ИМ       |7       |ШТ     |12/2021|0      |300     |5000  |718,15     |
|8507302009|KZ     |ЭК       |2       |ШТ     |12/2021|0      |4       |40000 |453,73     |
|3004490001|BY     |ИМ       |1       |1      |12/2021|0      |0       |66000 |4175,38    |
|9018390000|FR     |ИМ       |2       |ШТ     |12/2021|0      |91346   |40000 |310878,79  |
|7609000000|US     |ИМ       |7       |1      |12/2021|0      |0       |8000  |25957,06   |
|8536508009|CN     |ИМ       |2       |1      |12/2021|70     |0       |19000 |7

In [22]:
# dont let it write infinite partitioned csv
dfSample.coalesce(1).write.mode("overwrite").option("header", "true").csv("hdfs://hdfs-namenode:9000/spark/output/ingestedSample/")